In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import torch
import torchvision.transforms as T
import PIL.Image as Image

In [ ]:
input_model = "/home/mattia/Desktop/Repos/vggt/wrapper_output/vienna_state_opera/sparse"

import pycolmap
reconstruction = pycolmap.Reconstruction(input_model)
reconstruction.find_image_with_name("13/IMG_4697_frame_000001.jpg").cam_from_world

In [ ]:
from adjuster import Adjuster
adjuster = Adjuster(
    reconstruction = "/home/mattia/Desktop/Repos/vggt/wrapper_output/vienna_state_opera/sparse",
    images_path = "/home/mattia/Desktop/datasets/mydataset/data/vienna_state_opera/frames",
    depths_path = "/home/mattia/Desktop/Repos/vggt/wrapper_output/vienna_state_opera/sparse/depth_maps",
    single_camera_per_folder=True, 
    load_with_pad=False,
    lr=1e-3,
    grad_q=True,
    grad_t=True,
    grad_k=True,
)

adjuster.print_summary()

In [ ]:
out = adjuster(
    max_pairs=512,
    max_steps=20,
)

adjuster.build_reconstruction("/home/mattia/Desktop/Repos/batchsfm/optimized_reconstruction")

In [ ]:
from matplotlib import pyplot as plt
plt.plot(adjuster.loss_list)


In [ ]:
import sys
sys.path.append('/home/mattia/Desktop/Repos/wrapper_factory/benchmarks_3D')
from utils_benchmark_pose import *
from benchmark_pose import *

input = "/home/mattia/Desktop/Repos/vggt/wrapper_output/vienna_state_opera/sparse"
opt = "/home/mattia/Desktop/Repos/batchsfm/optimized_reconstruction"
target = "/home/mattia/Desktop/datasets/mydataset/data/vienna_state_opera/colmap/sparse/0"

AUC_score_max, num_images, df_initial = eval_colmap_model(input, target, return_df=True, thrs=[1,3,5])
th = 3
print("AUC:",AUC_score_max)
# print(sum(df_initial.q_error< th)/len(df_initial), sum(df_initial.t_error< th)/len(df_initial), sum(np.maximum(df_initial.q_error, df_initial.t_error)<th)/len(df_initial))
# df_initial.hist(["q_error", "t_error"], bins=100, range=(0, 10))

AUC_score_max, num_images, df_optim = eval_colmap_model(opt, target, return_df=True, thrs=[1,3,5])
th = 3
print("AUC:",AUC_score_max)
# print(sum(df_optim.q_error< th)/len(df_optim), sum(df_optim.t_error< th)/len(df_optim), sum(np.maximum(df_optim.q_error, df_optim.t_error)<th)/len(df_optim))
# df_optim.hist(["q_error", "t_error"], bins=100, range=(0, 10))

In [ ]:
.

In [ ]:
from mylib.plot import plot_imgs

print(len(viewgraph))
i,j = viewgraph[2000]
print(i, j)

i, j = '13/IMG_4697_frame_000001.jpg', '13/IMG_4697_frame_000003.jpg'

plot_imgs([images[i]['image'].permute(1,2,0).cpu(), images[j]['image'].permute(1,2,0).cpu()])

In [ ]:
plot_imgs([images[i]['edges_map'], images[j]['edges_map']], cmap="gray")

In [ ]:
plot_imgs([images[i]['depth'], images[j]['depth']], cmap="plasma")

In [ ]:
# plot_imgs([images[i]['dt_field'], images[j]['dt_field']], cmap="plasma")

In [ ]:
x1,y1,x2,y2,h,w = [int(x) for x in images[i]['coords']]
img1 = images[i]['image'][:, y1:y2, x1:x2]
img2 = images[j]['image'][:, y1:y2, x1:x2]
edge1 = images[i]['edges_map'][y1:y2, x1:x2]
edge2 = images[j]['edges_map'][y1:y2, x1:x2]
Z1 = images[i]['depth'][y1:y2, x1:x2][None]
Z2 = images[j]['depth'][y1:y2, x1:x2][None]

edge1.shape, edge2.shape, img1.shape, img2.shape, Z1.shape, Z2.shape

In [ ]:
from helpers.reprojection import compute_121_reprojection

data = {
    'P0': images[i]['P'].projection_matrix()[None], 
    'P1': images[j]['P'].projection_matrix()[None],
    'K0': intrinsics[images[i]['cam_id']].intrinsic_matrix()[None],  
    'K1': intrinsics[images[j]['cam_id']].intrinsic_matrix()[None],
    'depth0': Z1, 
    'depth1': Z2
}

kpts1, kpts2, tot = compute_121_reprojection(
    data,
    img1, img2,
    verbose=False, reprojection_error=3.0, border=5, sampling_factor=1)

print(f"Consistent points: {len(kpts1):,}, {len(kpts2):,} out of {tot:,} ({100*len(kpts1)/tot:.4f}%)")

In [ ]:
edge1_kpts = edge1.nonzero().flip(dims=(0,1))
edge2_kpts = edge2.nonzero().flip(dims=(0,1))

edge1_kpts.shape, edge2_kpts.shape

In [ ]:
from mylib.plot import plot_imgs_and_kpts

plot_imgs_and_kpts(
    edge1[..., None].repeat(1,1,3).cpu()*255//1,
    edge2[..., None].repeat(1,1,3).cpu()*255//1,
    edge1_kpts, edge2_kpts,
    sample_points=1_000, matches=False,
) 

In [ ]:
from losses.loss import sample_distance_field

dist = sample_distance_field(images[j]['dt_field'], images[i]['edges']).cpu()
print(f"Loss: {dist.mean():.4f}, dist length: {len(dist):,}, edges length: {len(images[i]['edges']):,}")

field = torch.zeros_like(images[j]['dt_field']).cpu()
for pt, d in zip(images[i]['edges'].long(), dist):
    field[pt[1], pt[0]] = d
plt.imshow(field, cmap="magma")

In [ ]:
for image_i, image_j in viewgraph[:10]:
    print(image_i, image_j)